In [1]:
import pandas as pd
import numpy as np
import requests
import os
import datetime as dt
import re
import ast

pd.set_option('display.max_columns', 999)

In [2]:
#Helper Functions

def convert_to_num(df):
    for col in df.columns:
        try:
            pd.to_numeric(df[col])
        except:
            print(f"Column {col} cannot be converted to numeric")
            continue

def labelDom(row):
    if row['DOM'] < 14:
        return True
    else:
        return False

def binaryDom(df):
    df['binaryDOM'] = df.apply(lambda row: labelDom(row), axis = 1)

def createDOM(df: pd.DataFrame, timestamp = False, convert = False):
    dom = df['soldDate'] - df['listDate']
    df['DOM'] = dom.astype("string")
    df.dropna(subset=['DOM'], inplace = True)
    df['DOM'] = df['DOM'].str.slice_replace(start=2)
    df['DOM'] = df['DOM'].astype("int")

def dfRemoveNA(df):
    df = df.replace('', np.nan)
    if sum(df.isna().sum()) != 0:
        df = df.dropna()
    return df

def dfFillNA(df, column):
    df = df.replace('', np.nan)
    df = df.fillna(value = {column: "No "+column})
    return df

def dfExtractFeatures(df: pd.DataFrame, key: str, columns: list):
    if type(df[key].values[0]) == str:
        serie = df[key].apply(lambda x: ast.literal_eval(str(x)))
        d = pd.DataFrame.from_dict(serie.tolist())
    elif type(df[key].values[0]) == dict:
        serie = pd.Series.to_dict(df[key])
        d = pd.DataFrame.from_dict(serie, orient='index')
    for i in range(len(columns)):
        df[columns[i]] = d[columns[i]]

    return df

def dfPriceDifference(df: pd.DataFrame):
    #should only be for clustering purposes only
    df['PriceDifference'] = df['soldPrice'] - df['listPrice']

def dfDatesRemoveTimeStamp(df: pd.DataFrame):
    df['listDate'] = df['listDate'].str.slice_replace(start = 10, repl="")
    df['soldDate'] = df['soldDate'].str.slice_replace(start = 10, repl="")

def dfConvertToDatetime(df: pd.DataFrame, timestamp = False):
    if type(df['listDate'].values[0]) == str:
        dfDatesRemoveTimeStamp(df)
    df['listDate'] = pd.to_datetime(df['listDate'])
    df['soldDate'] = pd.to_datetime(df['soldDate'])

def removeFeatures(df: pd.DataFrame, columns: list):
    #df = df.drop(['images', 'resource', 'timestamps','permissions','lastStatus','status','boardId','agents'], axis = 1)
    df = df.drop(columns, axis = 1)
    return df

#Data Augmentation

def countLevels(df: pd.DataFrame, column):
    series = df[column].value_counts()
    return series

def augmentCategorical(df, column, percent = 0.04):
    series = countLevels(df, column)
    total = np.sum(series.values)
    percentage = [x/total for x in series.values]
    
    for i in range(len(percentage)):
        if percentage[i] > percent:
            pass
        else:
            df = df.replace(to_replace = {column: series.index[i]}, value = 'Other '+column)

    return df

def augmentType(df, type, columns):
    for col in columns:
        df[col] = df[col].astype(df[col].astype(type))

    return df

def augmentNumericals(df):
    for i in range(len(df.columns)):
        if (type(df[df.columns[i]].values[0]) == str):
            numerical = re.search('^[-+]?[0-9]*\.?[0-9]+$', df[df.columns[i]].values[0])
            if numerical is not None:
                df[df.columns[i]] = df[df.columns[i]].astype(float)
    return df

def removeLowLevels(df, column):
    lst = []
    for i in range(len(df[column].value_counts().index)):
        if df[column].value_counts().values[i] < 11:
            lst.append(df[column].value_counts().index[i])

    for i in range(len(lst)):
        df = df[df[column] != lst[i]]
    return df

def removeOutliers(df):
    percentile90 = df['soldPrice'].quantile(0.90)
    percentile10 = df['soldPrice'].quantile(0.10)

    iqr = percentile90 - percentile10

    upper_limit = percentile90 + 3 * iqr
    lower_limit = percentile10 - 3 * iqr

    # df[df['soldPrice'] > upper_limit]
    # df[df['soldPrice'] < lower_limit]

    df = df[df['soldPrice'] < upper_limit]
    df = df[df['soldPrice'] > lower_limit]

    return df


In [3]:
raw_lease_df = pd.read_csv(os.getcwd()+"\\data\\raw_lease_data.csv")

In [4]:
raw_lease_df.columns

Index(['listDate', 'images', 'address', 'soldDate', 'resource', 'timestamps',
       'type', 'mlsNumber', 'permissions', 'soldPrice', 'details', 'class',
       'map', 'listPrice', 'lastStatus', 'status', 'boardId', 'agents'],
      dtype='object')

In [5]:
pd.unique(raw_lease_df['lastStatus'].values)

array(['Lsd', 'Exp', 'Sus', 'Ter', 'Lc', 'Sld'], dtype=object)

In [6]:
raw_lease_df.head()

,listDate,images,address,soldDate,resource,timestamps,type,mlsNumber,permissions,soldPrice,details,class,map,listPrice,lastStatus,status,boardId,agents
0,2019-04-16T00:00:00.000Z,"['IMG-N4418450_1.jpg', 'IMG-N4418450_2.jpg', '...","{'area': 'York', 'zip': 'L3Y0E1', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T10:56:42.000Z'},Lease,N4418450,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2200.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.05193999999999', 'longitude':...",2100.0,Lsd,U,2,[]
1,2019-05-01T00:00:00.000Z,"['IMG-N4435010_1.jpg', 'IMG-N4435010_2.jpg', '...","{'area': 'York', 'zip': 'L4H1T3', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T15:58:16.000Z'},Lease,N4435010,"{'displayAddressOnInternet': 'Y', 'displayPubl...",1350.0,"{'numBathrooms': '1', 'numBedrooms': '2', 'sqf...",ResidentialProperty,"{'latitude': '43.8168699', 'longitude': '-79.6...",1350.0,Lsd,U,2,[]
2,2019-05-03T00:00:00.000Z,"['IMG-N4437661_1.jpg', 'IMG-N4437661_2.jpg', '...","{'area': 'York', 'zip': 'L4E0T9', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-07T11:06:02.000Z'},Lease,N4437661,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3000.0,"{'numBathrooms': '4', 'numBedrooms': '5', 'sqf...",ResidentialProperty,"{'latitude': '43.9175319', 'longitude': '-79.4...",2999.0,Lsd,U,2,[]
3,2019-04-03T00:00:00.000Z,"['IMG-N4405561_1.jpg', 'IMG-N4405561_2.jpg', '...","{'area': 'York', 'zip': 'L9N 0T3', 'country': ...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T14:23:19.000Z'},Lease,N4405561,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.09185', 'longitude': '-79.501...",2400.0,Lsd,U,2,[]
4,2019-04-24T00:00:00.000Z,['IMG-W4426017_1.jpg'],"{'area': 'Peel', 'zip': 'L4X4H2', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-09T13:23:15.000Z'},Lease,W4426017,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '43.6064424', 'longitude': '-79.6...",3400.0,Lsd,U,2,[]


The datatypes are mainly objects, but there are dictionaries and floats/integers that are located inside each objects that may need to be converted. Certain predictors like number of bathrooms can be left as strings, 

In [7]:
raw_lease_df.dtypes

listDate        object
images          object
address         object
soldDate        object
resource        object
timestamps      object
type            object
mlsNumber       object
permissions     object
soldPrice      float64
details         object
class           object
map             object
listPrice      float64
lastStatus      object
status          object
boardId          int64
agents          object
dtype: object

In [8]:
address = ['area', 'city', 'district', 'neighborhood']
details = ['numBathrooms','numBedrooms','sqft','style',]
map_ = ['latitude', 'longitude']
columns = ['listDate','soldDate','address','details','images', 'resource', 'timestamps','permissions','lastStatus','status','boardId','agents','type','mlsNumber','map']

lease_df = dfExtractFeatures(raw_lease_df, key = 'address', columns = address)
lease_df = dfExtractFeatures(lease_df, key = 'details', columns = details)
lease_df = dfExtractFeatures(lease_df, key = 'map', columns = map_)

In [9]:
lease_df.head()

,listDate,images,address,soldDate,resource,timestamps,type,mlsNumber,permissions,soldPrice,details,class,map,listPrice,lastStatus,status,boardId,agents,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude
0,2019-04-16T00:00:00.000Z,"['IMG-N4418450_1.jpg', 'IMG-N4418450_2.jpg', '...","{'area': 'York', 'zip': 'L3Y0E1', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T10:56:42.000Z'},Lease,N4418450,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2200.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.05193999999999', 'longitude':...",2100.0,Lsd,U,2,[],York,Newmarket,Newmarket,Glenway Estates,4,4,1500-2000,3-Storey,44.05193999999999,-79.49083999999999
1,2019-05-01T00:00:00.000Z,"['IMG-N4435010_1.jpg', 'IMG-N4435010_2.jpg', '...","{'area': 'York', 'zip': 'L4H1T3', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T15:58:16.000Z'},Lease,N4435010,"{'displayAddressOnInternet': 'Y', 'displayPubl...",1350.0,"{'numBathrooms': '1', 'numBedrooms': '2', 'sqf...",ResidentialProperty,"{'latitude': '43.8168699', 'longitude': '-79.6...",1350.0,Lsd,U,2,[],York,Vaughan,Vaughan,Sonoma Heights,1,2,,2-Storey,43.8168699,-79.618265
2,2019-05-03T00:00:00.000Z,"['IMG-N4437661_1.jpg', 'IMG-N4437661_2.jpg', '...","{'area': 'York', 'zip': 'L4E0T9', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-07T11:06:02.000Z'},Lease,N4437661,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3000.0,"{'numBathrooms': '4', 'numBedrooms': '5', 'sqf...",ResidentialProperty,"{'latitude': '43.9175319', 'longitude': '-79.4...",2999.0,Lsd,U,2,[],York,Richmond Hill,Richmond Hill,Jefferson,4,5,3000-3500,2-Storey,43.9175319,-79.43661039999999
3,2019-04-03T00:00:00.000Z,"['IMG-N4405561_1.jpg', 'IMG-N4405561_2.jpg', '...","{'area': 'York', 'zip': 'L9N 0T3', 'country': ...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T14:23:19.000Z'},Lease,N4405561,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.09185', 'longitude': '-79.501...",2400.0,Lsd,U,2,[],York,East Gwillimbury,East Gwillimbury,Holland Landing,4,4,3000-3500,2-Storey,44.09185,-79.50193
4,2019-04-24T00:00:00.000Z,['IMG-W4426017_1.jpg'],"{'area': 'Peel', 'zip': 'L4X4H2', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-09T13:23:15.000Z'},Lease,W4426017,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '43.6064424', 'longitude': '-79.6...",3400.0,Lsd,U,2,[],Peel,Mississauga,Mississauga,Hurontario,4,4,,2-Storey,43.6064424,-79.6422499


In [10]:
lease_df.columns

Index(['listDate', 'images', 'address', 'soldDate', 'resource', 'timestamps',
       'type', 'mlsNumber', 'permissions', 'soldPrice', 'details', 'class',
       'map', 'listPrice', 'lastStatus', 'status', 'boardId', 'agents', 'area',
       'city', 'district', 'neighborhood', 'numBathrooms', 'numBedrooms',
       'sqft', 'style', 'latitude', 'longitude'],
      dtype='object')

Feature Engineering

In [11]:
pd.unique(lease_df['sqft'].values)

array(['1500-2000', '', '3000-3500', '2500-3000', '700-1100', '2000-2500',
       '3500-5000', '600-699', '1100-1500', '1200-1399', '500-599',
       '900-999', '800-899', '1400-1599', '700-799', '1000-1199',
       '1600-1799', '2000-2249', '0-499', '1800-1999', '< 700',
       '2250-2499', '5000+', '3500-3749', '500-699', '2750-2999',
       '3250-3499', '4000-4249', '2500-2749', '3000-3249', '1100-1299',
       '700-899', '3750-3999', '1300-1499', '900-1099', '4500-4749',
       '4750-4999', '4250-4499'], dtype=object)

In [12]:
lease_df.replace({"": np.nan}, inplace = True)
lease_df.replace({float("nan"): np.nan}, inplace = True)

In [13]:
dfConvertToDatetime(lease_df)

In [14]:
createDOM(lease_df)

In [15]:
lease_df['avg_sqft'] = [(int(value.split("-")[0])+int(value.split("-")[1]))/2 if not isinstance(value, float) and '-' in value else np.nan for value in lease_df['sqft']]

In [16]:
lease_df['avg_sqft']

0         1750.0
1            NaN
2         3250.0
3         3250.0
4            NaN
           ...  
320766     949.5
320767       NaN
320768     549.5
320769     649.5
320770     749.5
Name: avg_sqft, Length: 320770, dtype: float64

In [17]:
lease_df['ppsqft'] = lease_df['listPrice']/lease_df['avg_sqft']

In [18]:
lease_df['ppsqft']

0         1.200000
1              NaN
2         0.922769
3         0.738462
4              NaN
            ...   
320766    4.739336
320767         NaN
320768    4.549591
320769    4.926867
320770    3.202135
Name: ppsqft, Length: 320770, dtype: float64

In [19]:
pd.unique(lease_df['numBedrooms'].values)

array(['4', '2', '5', '3', '1', '0', '6', nan, '7', '8'], dtype=object)

In [20]:
lease_df['bathtobed_ratio'] = lease_df['numBathrooms'].astype("Int64")/lease_df['numBedrooms'].astype("Int64")

In [21]:
lease_df['latitude'] = lease_df['latitude'].astype("float")
lease_df['longitude'] = lease_df['longitude'].astype("float")

In [22]:
lease_df.tail()

,listDate,images,address,soldDate,resource,timestamps,type,mlsNumber,permissions,soldPrice,details,class,map,listPrice,lastStatus,status,boardId,agents,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude,DOM,avg_sqft,ppsqft,bathtobed_ratio
320766,2023-03-14,"['IMG-W5965967_1.jpg', 'IMG-W5965967_2.jpg', '...","{'area': 'Toronto', 'zip': 'M6S 1A1', 'country...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T21:21:01.000Z'},Lease,W5965967,"{'displayAddressOnInternet': 'Y', 'displayPubl...",4500.0,"{'numBathrooms': '3', 'numBedrooms': '3', 'sqf...",CondoProperty,"{'latitude': '43.6357626', 'longitude': '-79.4...",4500.0,Lsd,U,2,[],Toronto,Toronto,Toronto W01,High Park-Swansea,3,3,900-999,Apartment,43.635763,-79.467490,35,949.5,4.739336,1.0
320767,2023-03-23,"['IMG-W5985741_1.jpg', 'IMG-W5985741_2.jpg', '...","{'area': 'Toronto', 'zip': 'M6P2G3', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T17:01:48.000Z'},Lease,W5985741,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3700.0,"{'numBathrooms': '1', 'numBedrooms': '2', 'sqf...",ResidentialProperty,"{'latitude': '43.6607634', 'longitude': '-79.4...",3550.0,Lsd,U,2,[],Toronto,Toronto,Toronto W02,High Park North,1,2,NaN,2-Storey,43.660763,-79.460102,26,NaN,NaN,0.5
320768,2023-03-20,"['IMG-C5975287_1.jpg', 'IMG-C5975287_2.jpg', '...","{'area': 'Toronto', 'zip': 'M2J1M5', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T22:13:37.000Z'},Lease,C5975287,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2500.0,"{'numBathrooms': '1', 'numBedrooms': '1', 'sqf...",CondoProperty,"{'latitude': '43.7716467', 'longitude': '-79.3...",2500.0,Lsd,U,2,[],Toronto,Toronto,Toronto C15,Henry Farm,1,1,500-599,Apartment,43.771647,-79.344663,29,549.5,4.549591,1.0
320769,2023-04-13,"['IMG-C6024761_1.jpg', 'IMG-C6024761_2.jpg', '...","{'area': 'Toronto', 'zip': 'M5H0B1', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T22:49:33.000Z'},Lease,C6024761,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3500.0,"{'numBathrooms': '1', 'numBedrooms': '1', 'sqf...",CondoProperty,"{'latitude': '43.6505301', 'longitude': '-79.3...",3200.0,Lsd,U,2,[],Toronto,Toronto,Toronto C01,Bay Street Corridor,1,1,600-699,Apartment,43.650530,-79.382148,5,649.5,4.926867,1.0
320770,2023-04-04,"['IMG-S6014669_1.jpg', 'IMG-S6014669_2.jpg', '...","{'area': 'Simcoe', 'zip': 'L4M 6H9', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T07:27:53.000Z'},Lease,S6014669,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2400.0,"{'numBathrooms': '1', 'numBedrooms': '1', 'sqf...",CondoProperty,"{'latitude': '44.3896491', 'longitude': '-79.6...",2400.0,Lsd,U,2,[],Simcoe,Barrie,Barrie,North Shore,1,1,700-799,Apartment,44.389649,-79.684525,14,749.5,3.202135,1.0


In [23]:
44.389649	-79.684525

-35.294875999999995

In [24]:
top_left = [44.389649+0.05, -79.684525-0.05]
top_right = [44.389649+0.05, -79.684525+0.05]
bottom_right = [44.389649-0.05, -79.684525+0.05]
bottom_left = [44.389649-0.05, -79.684525-0.05]

coords = [[top_left, top_right, bottom_right, bottom_left, top_left]]

In [25]:
coords = [[top_left, top_right, bottom_right, bottom_left, top_left]]

In [26]:
coords

[[[44.439648999999996, -79.73452499999999],
  [44.439648999999996, -79.634525],
  [44.339649, -79.634525],
  [44.339649, -79.73452499999999],
  [44.439648999999996, -79.73452499999999]]]

In [27]:
coords = [[[-79.14121,43.79041],[-79.132627,43.773059],[-79.188932,43.886988],[-79.200605,43.877832],[-79.236654,43.869665],[-79.265836,43.860011],[-79.281972,43.856051],[-79.322828,43.84689],[-79.368146,43.839214],[-79.386021,43.836139],[-79.41486,43.838616],[-79.423787,43.836635],[-79.475285,43.82227],[-79.480092,43.813352],[-79.480778,43.803441],[-79.485585,43.79799],[-79.493825,43.794025],[-79.556996,43.779649],[-79.601628,43.761303],[-79.61611,43.758572],[-79.629934,43.750141],[-79.625471,43.728064],[-79.616888,43.713177],[-79.606245,43.695555],[-79.601095,43.685873],[-79.593885,43.681156],[-79.590109,43.672465],[-79.582212,43.671224],[-79.574659,43.670975],[-79.535177,43.58325],[-79.424627,43.619052],[-79.385488,43.602645],[-79.315451,43.612092],[-79.14121,43.79041]]]

In [28]:
def get_radius(lat, long, value = None):
    if value:
        return (lat + value, lat - value, long + value, long - value)
    else:
        return (lat + 0.005, lat - 0.005, long + 0.005, long - 0.005)
    
    #entire list[kings square:[[lat, long], [lat long], [] ,[]],[[], [], [] ,[]]]

def avg_radius(df):
    avg_prices = []
    for i in range(len(df)):
        avg_price = []
        radius = get_radius(df['latitude'].values[i], df['longitude'].values[i])
        for j in range(len(df)):
            if j != i:
                if (radius[0] <= df['latitude'].values[j] <= radius[1]) and (radius[2] <=  df['longitude'].values[j] <= radius[3]):
                    avg_price.append(df['listPrice'].values[j])
        if len(avg_price) == 0:
            avg_prices.append(np.nan)
        else:
            avg_prices.append(np.mean(avg_price))

    print(avg_prices)

    return avg_prices

def get_radius_(mlsNumber, api_key):
    url = f"https://api.repliers.io/listings/{mlsNumber}?similar?radius=1"

    payload = {}
    headers = {'repliers-api-key': api_key}
    r = requests.request("GET",url, params=payload, headers=headers)
    data = r.json()
    df = pd.DataFrame(data['listings'])
    return df

In [74]:
url = "https://api.repliers.io/listings/W5965967"
payload = {}
#payload = {'mlsNumber': "W5965967", 'status': 'U', 'lastStatus': 'Lsd',"resultsPerPage": 10000}
headers = {'repliers-api-key': os.environ['REPLIERS_KEY']}
r = requests.request("GET",url, params=payload, headers=headers)
print(r)
dat2 = r.json()

<Response [200]>


In [72]:
dat2

{'listDate': '2023-03-14T00:00:00.000Z',
 'rooms': {'1': {'features': 'Laminate',
   'level': 'Flat',
   'length': '6.10',
   'width': '3.25',
   'description': 'Living',
   'features3': 'W/O To Balcony',
   'features2': 'Combined W/Dining'},
  '2': {'features': 'Laminate',
   'level': 'Flat',
   'length': '6.10',
   'width': '3.25',
   'description': 'Dining',
   'features3': 'Combined W/Living',
   'features2': 'Open Concept'},
  '3': {'features': 'Open Concept',
   'level': 'Flat',
   'length': '6.10',
   'width': '3.25',
   'description': 'Kitchen',
   'features3': 'Quartz Counter',
   'features2': 'Stainless Steel Appl'},
  '4': {'features': 'Laminate',
   'level': 'Flat',
   'length': '3.05',
   'width': '3.05',
   'description': 'Prim Bdrm',
   'features3': '4 Pc Ensuite',
   'features2': 'His/Hers Closets'},
  '5': {'features': 'Laminate',
   'level': 'Flat',
   'length': '3.05',
   'width': '2.80',
   'description': '2nd Br',
   'features3': 'Large Window',
   'features2': 'La

In [68]:
import json

print(lease_df['mlsNumber'].values[-4])

def get_avg_comparables(df):
    mlsNumbers = df['mlsNumber'].values[-5]
    avg_prices = []
    url = f"https://api.repliers.io/listings?mlsNumber=W5965967&mlsNumber=W5985741&status=U&lastStatus=Lsd"
    #url = f"https://api.repliers.io/listings/"
    payload = {}
    #payload = {'mlsNumber': "W5965967", 'status': 'U', 'lastStatus': 'Lsd',"resultsPerPage": 10000}
    headers = {'repliers-api-key': os.environ['REPLIERS_KEY']}
    r = requests.request("GET",url, params=payload, headers=headers)
    print(r)
    data = r.json()

    return data, r
    #data = json.loads(r.text)
    #     for value in data['comparables']:
    #         avg_price.append(float(value['listPrice']))

    #     if len(avg_price) == 0:
    #         avg_prices.append(np.nan)
    #     else:
    #         print(np.mean(avg_price))

    # return avg_prices

W5985741


In [69]:
data, r = get_avg_comparables(lease_df)

<Response [200]>


In [70]:
data

{'page': 1,
 'numPages': 1,
 'pageSize': 100,
 'count': 2,
 'statistics': {'listPrice': {'min': 3550, 'max': 4500}},
 'listings': [{'listDate': '2023-03-14T00:00:00.000Z',
   'rooms': {'1': {'features': 'Laminate',
     'level': 'Flat',
     'length': '6.10',
     'width': '3.25',
     'description': 'Living',
     'features3': 'W/O To Balcony',
     'features2': 'Combined W/Dining'},
    '2': {'features': 'Laminate',
     'level': 'Flat',
     'length': '6.10',
     'width': '3.25',
     'description': 'Dining',
     'features3': 'Combined W/Living',
     'features2': 'Open Concept'},
    '3': {'features': 'Open Concept',
     'level': 'Flat',
     'length': '6.10',
     'width': '3.25',
     'description': 'Kitchen',
     'features3': 'Quartz Counter',
     'features2': 'Stainless Steel Appl'},
    '4': {'features': 'Laminate',
     'level': 'Flat',
     'length': '3.05',
     'width': '3.05',
     'description': 'Prim Bdrm',
     'features3': '4 Pc Ensuite',
     'features2': 'His/H

In [ ]:
r.url

'https://api.repliers.io/listings/?mlsNumber=N4418450&mlsNumber=N4435010&mlsNumber=N4437661&mlsNumber=N4405561&mlsNumber=W4426017&mlsNumber=C4432900&mlsNumber=W4356174&mlsNumber=W4424829&mlsNumber=N4424863&mlsNumber=N4406614&mlsNumber=N4425486&mlsNumber=E4424324&mlsNumber=C4433947&mlsNumber=W4424200&mlsNumber=X4439875&mlsNumber=E4430512&mlsNumber=X4430401&mlsNumber=N4423281&mlsNumber=N4422493&mlsNumber=N4429780&mlsNumber=W4438444&mlsNumber=W4437823&mlsNumber=E4425014&mlsNumber=C4425805&mlsNumber=N4429957&mlsNumber=N4437409&mlsNumber=C4438129&mlsNumber=N4437031&mlsNumber=N4438257&mlsNumber=N4438687&mlsNumber=W4438387&mlsNumber=W4414869&mlsNumber=S4414823&mlsNumber=E4413211&mlsNumber=N4435193&mlsNumber=W4436787&mlsNumber=C4436013&mlsNumber=N4434092&mlsNumber=E4434010&mlsNumber=X4436215&mlsNumber=E4435867&mlsNumber=N4435409&mlsNumber=E4435443&mlsNumber=N4431597&mlsNumber=S4432910&mlsNumber=W4435020&mlsNumber=N4433302&mlsNumber=W4432322&mlsNumber=N4432353&mlsNumber=W4428780&mlsNumber=N4431

In [ ]:
data

rough

In [ ]:
url = "https://api.repliers.io/listings?map=%5B%5B%5B44.439648999999996%2C%20-79.73452499999999%5D%2C%0A%20%20%5B44.439648999999996%2C%20-79.634525%5D%2C%0A%20%20%5B44.339649%2C%20-79.634525%5D%2C%0A%20%20%5B44.339649%2C%20-79.73452499999999%5D%2C%0A%20%20%5B44.439648999999996%2C%20-79.73452499999999%5D%5D%5D"

In [ ]:
outlst = [([str(c) for c in top_left])]

def turn_string(lst):
    return [str(c) for c in lst]

In [ ]:
coords = [[[-79.14121,43.79041],[-79.132627,43.773059],[-79.188932,43.886988],[-79.200605,43.877832],[-79.236654,43.869665],[-79.265836,43.860011],[-79.281972,43.856051],[-79.322828,43.84689],[-79.368146,43.839214],[-79.386021,43.836139],[-79.41486,43.838616],[-79.423787,43.836635],[-79.475285,43.82227],[-79.480092,43.813352],[-79.480778,43.803441],[-79.485585,43.79799],[-79.493825,43.794025],[-79.556996,43.779649],[-79.601628,43.761303],[-79.61611,43.758572],[-79.629934,43.750141],[-79.625471,43.728064],[-79.616888,43.713177],[-79.606245,43.695555],[-79.601095,43.685873],[-79.593885,43.681156],[-79.590109,43.672465],[-79.582212,43.671224],[-79.574659,43.670975],[-79.535177,43.58325],[-79.424627,43.619052],[-79.385488,43.602645],[-79.315451,43.612092],[-79.14121,43.79041]]]

In [ ]:
# top_left = [44.389649+0.05, -79.684525-0.05]
# top_right = [44.389649+0.05, -79.684525+0.05]
# bottom_right = [44.389649-0.05, -79.684525+0.05]
# bottom_left = [44.389649-0.05, -79.684525-0.05]

#coords = [[top_left, top_right, bottom_right, bottom_left, top_left]]

url = f"https://api.repliers.io/listings/"
#payload = {"map": coords, "status": 'U'}
payload = {
"map": [
[
[
-79.39010775366981,
43.65576424783316
],
[
-79.39357890073602,
43.64856703742666
],
[
-79.38311365137208,
43.647667325480626
],
[
-79.37440987962434,
43.65059134003147
],
[
-79.3750315776063,
43.65969985745275
],
[
-79.38674022293412,
43.661798743774085
],
[
-79.39010775366981,
43.65576424783316
]
]
]
}
headers = {'repliers-api-key': os.environ['REPLIERS_KEY']}
r = requests.request("POST",url, params=payload, headers=headers)
#print(r)
data = r.json()

In [ ]:
r.url

'https://api.repliers.io/listings/?map=%5B%2744.439648999999996%27%2C+%27-79.73452499999999%27%5D&map=%5B%2744.439648999999996%27%2C+%27-79.634525%27%5D&map=%5B%2744.339649%27%2C+%27-79.634525%27%5D&map=%5B%2744.339649%27%2C+%27-79.73452499999999%27%5D&map=%5B%2744.439648999999996%27%2C+%27-79.73452499999999%27%5D'

In [ ]:
data

In [ ]:
numbers = ['W5965967', 'W5985741', 'C5975287', 'C6024761', 'S6014669']

In [ ]:
r.url

'https://api.repliers.io/listings/?map=%5B%2744.439648999999996%27%2C+%27-79.73452499999999%27%5D&map=%5B%2744.439648999999996%27%2C+%27-79.634525%27%5D&map=%5B%2744.339649%27%2C+%27-79.634525%27%5D&map=%5B%2744.339649%27%2C+%27-79.73452499999999%27%5D&map=%5B%2744.439648999999996%27%2C+%27-79.73452499999999%27%5D'

In [ ]:
url = f"https://api.repliers.io/listings/"
payload = {"mlsNumber": numbers, "status": ["U"], "lastStatus": "Lsd"}
payload = {}
headers = {'repliers-api-key': os.environ['REPLIERS_KEY']}
r = requests.request("GET",url, params=payload, headers=headers)
#print(r)
data = r.json()

data

In [ ]:
url = f"https://api.repliers.io/listings/"
payload = {"mlsNumber": numbers, "status": ["U"], "lastStatus": "Lsd"}
headers = {'repliers-api-key': os.environ['REPLIERS_KEY']}
r = requests.request("GET",url, params=payload, headers=headers)
#print(r)
data = r.json()

In [ ]:
r.url

'https://api.repliers.io/listings/?mlsNumber=W5965967&mlsNumber=W5985741&mlsNumber=C5975287&mlsNumber=C6024761&mlsNumber=S6014669&status=U'

In [ ]:
import requests
import json
url = "https://api.repliers.io/listings"
headers = {
"repliers-api-key": os.environ['REPLIERS_KEY'],
"Content-Type": "application/json"
}
data = {
"map": 
coords
}
response = requests.request("GET",url, headers=headers, params=json.dumps(data))
print(response.status_code)
print(response.json())

200
{'page': 1, 'numPages': 432, 'pageSize': 100, 'count': 43160, 'statistics': {'listPrice': {'min': 0, 'max': 95000000}}, 'listings': [{'listDate': '2023-12-16T00:00:00.000Z', 'rooms': {'1': {'features': '', 'level': '', 'length': '', 'width': '', 'description': '', 'features3': '', 'features2': ''}, '2': {'features': '', 'level': '', 'length': '', 'width': '', 'description': '', 'features3': '', 'features2': ''}, '3': {'features': '', 'level': '', 'length': '', 'width': '', 'description': '', 'features3': '', 'features2': ''}, '4': {'features': '', 'level': '', 'length': '', 'width': '', 'description': '', 'features3': '', 'features2': ''}, '5': {'features': '', 'level': '', 'length': '', 'width': '', 'description': '', 'features3': '', 'features2': ''}, '6': {'features': '', 'level': '', 'length': '', 'width': '', 'description': '', 'features3': '', 'features2': ''}, '7': {'features': '', 'level': '', 'length': '', 'width': '', 'description': '', 'features3': '', 'features2': ''}, 

In [ ]:
json.dumps(data)

'{"map": [[[44.439648999999996, -79.73452499999999], [44.439648999999996, -79.634525], [44.339649, -79.634525], [44.339649, -79.73452499999999], [44.439648999999996, -79.73452499999999]]]}'

In [ ]:
sale = pd.read_csv(os.getcwd()+"\\data\\raw_sale_data.csv")

In [ ]:
len(sale)

514720

In [ ]:
import asyncio

lst = [1,2,3,4,5,6,7,8,9]

async def counta(lst):
    final = []
    for i in range(len(lst)):
        print(lst[i])
        await asyncio.sleep(1)
        print(lst[i]/2)
        final.append(lst[i]/2)
    return final

async def main():
    final = await asyncio.gather(counta(lst), counta(lst))
    return

final = await main()

1
1
0.5
2
0.5
2
1.0
3
1.0
3
1.5
4
1.5
4
2.0
5
2.0
5
2.5
6
2.5
6
3.0
7
3.0
7
3.5
8
3.5
8
4.0
9
4.0
9
4.5
4.5


In [ ]:
final

[[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5],
 [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]]

In [ ]:
async def grab_avg_prices(df):
    pass

In [ ]:
def test_get_avg_comparables(df):
    mlsNumbers = df['mlsNumber'].values[:400]
    avg_prices = []

    url = f"https://api.repliers.io/listings?resultsPerPage=10000"

    payload = {'mlsNumber': mlsNumbers, 'status': 'U', 'lastStatus': 'Lsd'}
    headers = {'repliers-api-key': os.environ['REPLIERS_KEY']}
    r = requests.request("GET",url, params=payload, headers=headers)
    print(r)
    data = r.json()
    return data

In [ ]:
data = test_get_avg_comparables(lease_df)

<Response [200]>


In [ ]:
data

{'page': 1,
 'numPages': 1,
 'pageSize': 10000,
 'count': 400,
 'statistics': {'listPrice': {'min': 1000, 'max': 12500}},
 'listings': [{'listDate': '2019-04-16T00:00:00.000Z',
   'images': ['IMG-N4418450_1.jpg',
    'IMG-N4418450_2.jpg',
    'IMG-N4418450_3.jpg',
    'IMG-N4418450_4.jpg',
    'IMG-N4418450_5.jpg',
    'IMG-N4418450_6.jpg',
    'IMG-N4418450_7.jpg',
    'IMG-N4418450_8.jpg',
    'IMG-N4418450_9.jpg'],
   'address': {'area': 'York',
    'zip': 'L3Y0E1',
    'country': None,
    'city': 'Newmarket',
    'streetNumber': '15',
    'unitNumber': '',
    'streetDirection': '',
    'streetName': 'Heswall',
    'district': 'Newmarket',
    'streetSuffix': 'Lane',
    'neighborhood': 'Glenway Estates',
    'state': 'Ontario',
    'majorIntersection': 'Yonge/ Davis',
    'communityCode': '09.07.0020'},
   'soldDate': '2019-05-06T00:00:00.000Z',
   'resource': 'Property:2381',
   'timestamps': {'listingUpdated': '2019-05-08T10:56:42.000Z'},
   'type': 'Lease',
   'mlsNumber': 'N4

In [ ]:
len(lease_df)

320770

In [ ]:
subarrays = []
for i in range(0, len(lease_df), 400):
    subarrays.append([lease_df.index[i:i+400]])
if len(subarrays[-1]) > len(lease_df):
    subarrays[-1] = [lease_df.index[subarrays[-1][0], lease_df[len(lease_df)]]]

subarrays

[[Int64Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
              ...
              390, 391, 392, 393, 394, 395, 396, 397, 398, 399],
             dtype='int64', length=400)],
 [Int64Index([400, 401, 402, 403, 404, 405, 406, 407, 408, 409,
              ...
              790, 791, 792, 793, 794, 795, 796, 797, 798, 799],
             dtype='int64', length=400)],
 [Int64Index([ 800,  801,  802,  803,  804,  805,  806,  807,  808,  809,
              ...
              1190, 1191, 1192, 1193, 1194, 1195, 1196, 1197, 1198, 1199],
             dtype='int64', length=400)],
 [Int64Index([1200, 1201, 1202, 1203, 1204, 1205, 1206, 1207, 1208, 1209,
              ...
              1590, 1591, 1592, 1593, 1594, 1595, 1596, 1597, 1598, 1599],
             dtype='int64', length=400)],
 [Int64Index([1600, 1601, 1602, 1603, 1604, 1605, 1606, 1607, 1608, 1609,
              ...
              1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999],
             dtype='int64', length=40

In [ ]:
import asyncio

async def count():
    data = [1, 2, 3, 4, 5]
    for x in data:
        print(x)
    print("sleeping")
    await asyncio.sleep(1)
    print("done sleeping")
    print("Two")

    return 1

async def count2(x):
    print("three")
    print(x+1)

    


async def main():
    await asyncio.gather(count(), count2())


final = await main()

1
2
3
4
5
sleeping
three


TypeError: unsupported operand type(s) for +: 'coroutine' and 'int'

done sleeping
Two


In [ ]:
import asyncio

async def process_x(x):
    print(f"Processing {x}")
    await asyncio.sleep(1)
    print(f"Finished processing {x}")
    return x * 2

async def process_y(x):
    print(f"Processing y on {x}")
    return x * 3

async def main():
    data = [1, 2, 3, 4, 5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]
    for x in data:
        # Create a task for process_x
        task_x = asyncio.create_task(process_x(x))

        # Run process_y immediately while process_x is sleeping
        result_y = process_y(x)
        print(f"Results for y on {x}: {result_y}")

        # Wait for process_x to complete
        result_x = await task_x
        print(f"Results for x: {result_x}")

# Run the main function using the asyncio event loop
await main()

Processing y on 1
Results for y on 1: 3
Processing 1
Finished processing 1
Results for x: 2
Processing y on 2
Results for y on 2: 6
Processing 2
Finished processing 2
Results for x: 4
Processing y on 3
Results for y on 3: 9
Processing 3
Finished processing 3
Results for x: 6
Processing y on 4
Results for y on 4: 12
Processing 4
Finished processing 4
Results for x: 8
Processing y on 5
Results for y on 5: 15
Processing 5
Finished processing 5
Results for x: 10
Processing y on 6
Results for y on 6: 18
Processing 6
Finished processing 6
Results for x: 12
Processing y on 7
Results for y on 7: 21
Processing 7
Finished processing 7
Results for x: 14
Processing y on 8
Results for y on 8: 24
Processing 8
Finished processing 8
Results for x: 16
Processing y on 9
Results for y on 9: 27
Processing 9
Finished processing 9
Results for x: 18
Processing y on 10
Results for y on 10: 30
Processing 10
Finished processing 10
Results for x: 20
Processing y on 11
Results for y on 11: 33
Processing 11
Finish

In [ ]:
import asyncio

async def process_x(x):
    print(f"grabbed {x}")
    print("sleeping")
    await asyncio.sleep(1)
    print("done sleeping")
    print(f"returning {x*2}")
    return x * 2

async def process_y(x):
    print(f"grabbed x returning y on {x}")
    return -x

async def main():
    data = [1, 2, 3, 4, 5]
    for x in data:
        # Create a task for process_x
        task_x = asyncio.create_task(process_x(x))
        
        # Run process_y immediately while process_x is sleeping

        result_x = await task_x

        result_y = process_y(result_x)
        print(f"Results for y on {x}: {result_y}")

        # Wait for process_x to complete
        #result_x = await task_x
        #print(f"Results for x: {result_x}")

# Run the main function using the asyncio event loop
await main()

grabbed 1
sleeping
done sleeping
returning 2
Results for y on 1: <coroutine object process_y at 0x000001B482B1A240>
grabbed 2
sleeping
done sleeping
returning 4
Results for y on 2: <coroutine object process_y at 0x000001B482B1A340>
grabbed 3
sleeping


C:\Users\Jackson\AppData\Local\Temp\ipykernel_36932\3058223667.py:25: RuntimeWarning: coroutine 'process_y' was never awaited
  result_y = process_y(result_x)


done sleeping
returning 6
Results for y on 3: <coroutine object process_y at 0x000001B482B1A2C0>
grabbed 4
sleeping
done sleeping
returning 8
Results for y on 4: <coroutine object process_y at 0x000001B482B1A240>
grabbed 5
sleeping
done sleeping
returning 10
Results for y on 5: <coroutine object process_y at 0x000001B482B1A340>


C:\Users\Jackson\AppData\Local\Temp\ipykernel_36932\3058223667.py:33: RuntimeWarning: coroutine 'process_y' was never awaited
  await main()


In [ ]:
import asyncio

async def task1():
    for x in [1,2,3,4,5]:
        print("Task 1 started")
        print(x)
        print("sleeping")
        await asyncio.sleep(3)
        print("done sleeping")
    result = "Task 1 completed"
    return result

async def task2():
    print("Task 2 started")
    task1_result = await task1()
    print("Task 2 received task1 result: ", task1_result)
    for i in range(5):
        print("Task 2 running")
        await asyncio.sleep(1)
    print("Task 2 completed")

async def main():
    # Run both tasks concurrently
    task2_coro = asyncio.create_task(task2())

    # Wait for task2 to complete
    await task2_coro

# Run the main function using the asyncio event loop
await main()

In [75]:
subarrays = []
for i in range(0, len(lease_df), 400):
    subarrays.append(lease_df.index[i:i+400])
if len(subarrays[-1]) > len(lease_df):
    subarrays[-1] = [lease_df.index[subarrays[-1][0], lease_df[len(lease_df)]]]

subarrays

[Int64Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
             ...
             390, 391, 392, 393, 394, 395, 396, 397, 398, 399],
            dtype='int64', length=400),
 Int64Index([400, 401, 402, 403, 404, 405, 406, 407, 408, 409,
             ...
             790, 791, 792, 793, 794, 795, 796, 797, 798, 799],
            dtype='int64', length=400),
 Int64Index([ 800,  801,  802,  803,  804,  805,  806,  807,  808,  809,
             ...
             1190, 1191, 1192, 1193, 1194, 1195, 1196, 1197, 1198, 1199],
            dtype='int64', length=400),
 Int64Index([1200, 1201, 1202, 1203, 1204, 1205, 1206, 1207, 1208, 1209,
             ...
             1590, 1591, 1592, 1593, 1594, 1595, 1596, 1597, 1598, 1599],
            dtype='int64', length=400),
 Int64Index([1600, 1601, 1602, 1603, 1604, 1605, 1606, 1607, 1608, 1609,
             ...
             1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999],
            dtype='int64', length=400),
 Int64Index([2000, 2

In [76]:
len(subarrays)

802

In [77]:
subarrays[-1][0]

320401

In [78]:
import asyncio
import queue

async def request_comparable_listings(q):
    print("Task 1 started")
    for lst in subarrays:
        #print(f"putting in {i}")
        await q.put(lst)
        print("sleeping")
        await asyncio.sleep(1)
        print("done sleeping")
    print("Task 1 completed")

# async def task2(q):
#     print("Task 2 started")
#     while True:
#         try:
#             value = q.get()
#             print("Task 2 received value:", value)
#         except queue.Empty():
#             print("Task 2 waiting for value...")
#             await asyncio.sleep(1)
#             continue
#         if value is None:
#             print("Task 2 completed")
#             break

async def get_average_price(q):
    print("Task 2 started")
    while True:
        print("Task 2 waiting for item...")
        value = await q.get()
        if value is None:
            print("Task 2 completed")
            break
        else:
            print("Task 2 received value:", value)
            print(value//2)


async def main():
    q = asyncio.Queue()
    # Run both tasks concurrently
    task1_coro = asyncio.create_task(request_comparable_listings(q))
    task2_coro = asyncio.create_task(get_average_price(q))

    # Wait for task1 to complete
    await task1_coro

    # Signal task2 to complete
    await q.put(None)
    await task2_coro

# Start the event loop and run the main coroutine
await main()


Task 1 started
sleeping
Task 2 started
Task 2 waiting for item...
Task 2 received value: Int64Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
            ...
            390, 391, 392, 393, 394, 395, 396, 397, 398, 399],
           dtype='int64', length=400)
Int64Index([  0,   0,   1,   1,   2,   2,   3,   3,   4,   4,
            ...
            195, 195, 196, 196, 197, 197, 198, 198, 199, 199],
           dtype='int64', length=400)
Task 2 waiting for item...
done sleeping
sleeping
Task 2 received value: Int64Index([400, 401, 402, 403, 404, 405, 406, 407, 408, 409,
            ...
            790, 791, 792, 793, 794, 795, 796, 797, 798, 799],
           dtype='int64', length=400)
Int64Index([200, 200, 201, 201, 202, 202, 203, 203, 204, 204,
            ...
            395, 395, 396, 396, 397, 397, 398, 398, 399, 399],
           dtype='int64', length=400)
Task 2 waiting for item...
done sleeping
sleeping
Task 2 received value: Int64Index([ 800,  801,  802,  803,  804,  805,  

CancelledError: 

In [ ]:
import asyncio
import queue

async def task1(q):
    print("Task 1 started")
    my_list = [1, 2, 3, 4, 5]
    for i in my_list:
        print(f"putting in {i}")
        await q.put(i)
        print("sleeping")
        await asyncio.sleep(3)
        print("done sleeping")
    print("Task 1 completed")

# async def task2(q):
#     print("Task 2 started")
#     while True:
#         try:
#             value = q.get()
#             print("Task 2 received value:", value)
#         except queue.Empty():
#             print("Task 2 waiting for value...")
#             await asyncio.sleep(1)
#             continue
#         if value is None:
#             print("Task 2 completed")
#             break

async def task2(q):
    print("Task 2 started")
    while True:
        print("Task 2 waiting for item...")
        value = await q.get()
        if value is None:
            print("Task 2 completed")
            break
        print("Task 2 received value:", value)


async def main():
    q = asyncio.Queue()
    # Run both tasks concurrently
    task1_coro = asyncio.create_task(task1(q))
    task2_coro = asyncio.create_task(task2(q))

    # Wait for task1 to complete
    await task1_coro

    # Signal task2 to complete
    await q.put(None)
    await task2_coro

# Start the event loop and run the main coroutine
await main()


Task 1 started
putting in 1
sleeping
Task 2 started
Task 2 waiting for item...
Task 2 received value: 1
Task 2 waiting for item...
done sleeping
putting in 2
sleeping
Task 2 received value: 2
Task 2 waiting for item...
done sleeping
putting in 3
sleeping
Task 2 received value: 3
Task 2 waiting for item...
done sleeping
putting in 4
sleeping
Task 2 received value: 4
Task 2 waiting for item...
done sleeping
putting in 5
sleeping
Task 2 received value: 5
Task 2 waiting for item...
done sleeping
Task 1 completed
Task 2 completed


In [84]:
import requests
import json
url = "https://api.repliers.io/listings/C6053387"
headers = {
"repliers-api-key": os.environ['REPLIERS_KEY'],
"Content-Type": "application/json"
}
payload = {}
response = requests.request("GET",url, headers=headers, params=payload)

data = response.json()

df = pd.DataFrame.from_dict(data, orient = 'index')
df = df.transpose()


In [59]:
df.columns

Index(['listDate', 'rooms', 'originalPrice', 'soldDate', 'occupancy',
       'timestamps', 'condominium', 'taxes', 'office', 'type', 'nearby', 'lot',
       'mlsNumber', 'openHouse', 'permissions', 'soldPrice', 'details',
       'class', 'map', 'images', 'address', 'comparables', 'resource', 'raw',
       'updatedOn', 'history', 'daysOnMarket', 'agents', 'coopCompensation',
       'listPrice', 'lastStatus', 'status', 'boardId'],
      dtype='object')

In [85]:
def dfExtractFeatures(df: pd.DataFrame, key: str, columns: list):
    if type(df[key].values[0]) == str:
        try:
            serie = df[key].apply(lambda x: ast.literal_eval(str(x)))
            d = pd.DataFrame.from_dict(serie.tolist())
        except:
            print("String does not match proper conversion methods")
    elif type(df[key].values[0]) == dict:
        serie = pd.Series.to_dict(df[key])
        d = pd.DataFrame.from_dict(serie, orient='index')
    for i in range(len(columns)):
        df[columns[i]] = d[columns[i]]

    return df

In [86]:
address = ['area', 'city', 'district', 'neighborhood']
details = ['numBathrooms','numBedrooms','sqft','style',]
map_ = ['latitude', 'longitude']

df = dfExtractFeatures(df, key = 'address', columns = address)
df = dfExtractFeatures(df, key = 'details', columns = details)
df = dfExtractFeatures(df, key = 'map', columns = map_)

In [87]:
df.columns

Index(['listDate', 'rooms', 'originalPrice', 'soldDate', 'occupancy',
       'timestamps', 'condominium', 'taxes', 'office', 'type', 'nearby', 'lot',
       'mlsNumber', 'openHouse', 'permissions', 'soldPrice', 'details',
       'class', 'map', 'images', 'address', 'comparables', 'resource', 'raw',
       'updatedOn', 'history', 'daysOnMarket', 'agents', 'coopCompensation',
       'listPrice', 'lastStatus', 'status', 'boardId', 'area', 'city',
       'district', 'neighborhood', 'numBathrooms', 'numBedrooms', 'sqft',
       'style', 'latitude', 'longitude'],
      dtype='object')

In [109]:
df['listDate'].values[0]

listDate = dt.datetime.strptime(df['listDate'].values[0].split("T")[0], '%Y-%m-%d')

In [89]:
import datetime as dt


In [90]:
def grab_average_area(df):
    url = f"https://api.repliers.io/listings?resultsPerPage=1000"
    payload = {'area': df['area'].values[0], 'type': df['type'].values[0], 'status': 'A'}
    headers = {'repliers-api-key': os.environ['REPLIERS_KEY']}
    r = requests.request("GET",url, params=payload, headers=headers)
    data = r.json()
    df_area = pd.DataFrame(data['listings'])
    columns = [col for col in df_area.columns if col != 'listPrice']
    df_area = df_area.drop(columns, axis = 1)
    df['listPrice_by_area'] = np.mean(df_area['listPrice'].astype("float").values)


def grab_average_city(df):
    url = f"https://api.repliers.io/listings?resultsPerPage=1000"
    payload = {'city': df['city'].values[0], 'type': df['type'].values[0], 'status': 'A'}
    headers = {'repliers-api-key': os.environ['REPLIERS_KEY']}
    r = requests.request("GET",url, params=payload, headers=headers)
    data = r.json()
    df_city = pd.DataFrame(data['listings'])
    columns = [col for col in df_city.columns if col != 'listPrice']
    df_city = df_city.drop(columns, axis = 1)
    df['listPrice_by_city'] = np.mean(df_city['listPrice'].astype("float").values)   

In [91]:
import pickle
sale_file = pickle.load(open(os.getcwd()+"\\models\\sale_model_parameters.sav", 'rb'))
lease_file = pickle.load(open(os.getcwd()+"\\models\\lease_model_parameters.sav", 'rb'))

sale_model = sale_file['model']
sale_columns = sale_file['columns']
sale_feat_index = sale_file['idx']

In [94]:
def feature_engineer_single_listing(df, model_columns, model_idx):
    #Replace any empty strings or nan values that are not specific to the pandas DataFrame
    df.replace({"": np.nan}, inplace = True)
    df.replace({float("nan"): np.nan}, inplace = True)

    df['listPrice'] = df['listPrice'].astype("float")
    df['soldPrice'] = df['soldPrice'].astype("float")
    
    #Replace any empty strings or nan values that are not specific to the pandas DataFrame
    df.replace({"": np.nan}, inplace = True)
    df.replace({float("nan"): np.nan}, inplace = True)
    
    #Encode map coordinates as floats
    df['latitude'] = df['latitude'].astype("float")
    df['longitude'] = df['longitude'].astype("float")

    #dom
    today = dt.datetime.now()
    if 'listDate' in df.columns:
        try:
            listDate = dt.datetime.strptime(df['listDate'].values[0].split("T")[0], '%Y-%m-%d')
            dom = (today-listDate).day
        except:
            dom = np.nan
    else:
        dom = np.nan 

    df['DOM'] = dom

    #bed bath ratio
    try:
        bedtobath = df['numBathrooms'].astype('float')/df['numBedrooms'].astype('float')
    except:
        bedtobath = np.nan
    
    df['bathtobed_ratio'] = bedtobath

    #sqft and price per sqft
    if '-' in df['sqft']:
        average_sqft = (int(df['sqft'].split("-")[0])+int(df['sqft'].split("-")[1]))/2
        df['avg_sqft'] = average_sqft

        pp_sqft = df['listPrice']/average_sqft
        df['ppsqft'] = pp_sqft
    else:
        avg_sqft = np.nan
        pp_sqft = np.nan

        df['avg_sqft'] = avg_sqft
        df['ppsqft'] = pp_sqft


    #average listing of city
    grab_average_city(df)

    #average listing of area
    grab_average_area(df)

    columns_to_keep = ['originalPrice', 'soldPrice', 'class', 'listPrice', 'area', 'city', 'district', 'neighborhood', 'numBathrooms', 'numBedrooms', 'sqft', 'style', 'latitude', 'longitude', 'DOM', 'avg_sqft', 'ppsqft', 'bathtobed_ratio', 'listPrice_by_area', 'listPrice_by_city']
    columns_to_drop = [col for col in df.columns if col not in columns_to_keep]
    df_agg = df.drop(columns = columns_to_drop, axis = 1)

    df_agg_cats = pd.get_dummies(df_agg)

    df_model_columns = pd.DataFrame(df_agg_cats, columns = model_columns)

    df_listing = pd.DataFrame(df_model_columns, columns = df_model_columns.columns[model_idx])

    return df_listing



In [98]:
df_ = feature_engineer_single_listing(df)

In [99]:
df_

,soldPrice,listPrice,sqft,latitude,longitude,DOM,bathtobed_ratio,avg_sqft,ppsqft,listPrice_by_city,listPrice_by_area,originalPrice_1999000.00,class_ResidentialProperty,area_Toronto,city_Toronto,district_Toronto C04,neighborhood_Bedford Park-Nortown,numBathrooms_2,numBedrooms_2,style_Bungalow
0,0.0,1999000.0,NaN,43.729052,-79.426974,NaN,1.0,NaN,NaN,1424204.466,1424204.466,1,1,1,1,1,1,1,1,1


In [100]:
df_ = pd.DataFrame(df_, columns = sale_columns)

In [101]:
df_

,listPrice,latitude,longitude,DOM,avg_sqft,ppsqft,bathtobed_ratio,listPrice_by_area,listPrice_by_city,originalPrice_100,originalPrice_100000000,originalPrice_101888800,originalPrice_101900000,originalPrice_102400000,originalPrice_102500000,originalPrice_102800000,originalPrice_102880000,originalPrice_102990000,originalPrice_103500000,originalPrice_103800000,originalPrice_103900000,originalPrice_103990000,originalPrice_104900000,originalPrice_104980000,originalPrice_104990000,originalPrice_104999900,originalPrice_105000000,originalPrice_105700000,originalPrice_105888800,originalPrice_106900000,originalPrice_106990000,originalPrice_107490000,originalPrice_107500000,originalPrice_107700000,originalPrice_108000000,originalPrice_108500000,originalPrice_108800000,originalPrice_108880000,originalPrice_108888800,originalPrice_108900000,originalPrice_109000000,originalPrice_109500000,originalPrice_109800000,originalPrice_109880000,originalPrice_109890000,originalPrice_109900000,originalPrice_109970000,originalPrice_109980000,originalPrice_109990000,originalPrice_109999000,originalPrice_109999900,originalPrice_110000000,originalPrice_110090000,originalPrice_112490000,originalPrice_112500000,originalPrice_112900000,originalPrice_112990000,originalPrice_113000000,originalPrice_114800000,originalPrice_114900000,originalPrice_114990000,originalPrice_114999900,originalPrice_115000000,originalPrice_115400000,originalPrice_115490000,originalPrice_115700000,originalPrice_115990000,originalPrice_116000000,originalPrice_116500000,originalPrice_116888800,originalPrice_116900000,originalPrice_116990000,originalPrice_117500000,originalPrice_117900000,originalPrice_118000000,originalPrice_118800000,originalPrice_118880000,originalPrice_118888800,originalPrice_118900000,originalPrice_119000000,originalPrice_119500000,originalPrice_119700000,originalPrice_119800000,originalPrice_119900000,originalPrice_119978600,originalPrice_119980000,originalPrice_119988800,originalPrice_119990000,originalPrice_119999000,originalPrice_119999900,originalPrice_120000000,originalPrice_122500000,originalPrice_122550000,originalPrice_122800000,originalPrice_123990000,originalPrice_124500000,originalPrice_124800000,originalPrice_124880000,originalPrice_124900000,originalPrice_124980000,originalPrice_124990000,originalPrice_124999000,originalPrice_125000000,originalPrice_125888800,originalPrice_125900000,originalPrice_126900000,originalPrice_127000000,originalPrice_127500000,originalPrice_127888500,originalPrice_127900000,originalPrice_127900100,originalPrice_127990000,originalPrice_127999900,originalPrice_128000000,originalPrice_128800000,originalPrice_128880000,originalPrice_128888800,originalPrice_128900000,originalPrice_128990000,originalPrice_129000000,originalPrice_129500000,originalPrice_129800000,originalPrice_129880000,originalPrice_12990000,originalPrice_129900000,originalPrice_129980000,originalPrice_129990000,originalPrice_129999900,originalPrice_130000000,originalPrice_131900000,originalPrice_132500000,originalPrice_132590000,originalPrice_132900000,originalPrice_132990000,originalPrice_133000000,originalPrice_133388800,originalPrice_133888800,originalPrice_133900000,originalPrice_134900000,originalPrice_134950000,originalPrice_134990000,originalPrice_134999900,originalPrice_135000000,originalPrice_135900000,originalPrice_136800000,originalPrice_136900000,originalPrice_136999900,originalPrice_137488800,originalPrice_137500000,originalPrice_137800000,originalPrice_137900000,originalPrice_137990000,originalPrice_138000000,originalPrice_138080000,originalPrice_138780000,originalPrice_138800000,originalPrice_138880000,originalPrice_138888800,originalPrice_138980000,originalPrice_13900000,originalPrice_139000000,originalPrice_139500000,originalPrice_139880000,originalPrice_139900000,originalPrice_139990000,originalPrice_139999900,originalPrice_140000000,originalPrice_141800000,originalPrice_142500000,originalPrice_142900000,originalPrice_143900000,originalPrice_1448

In [102]:
df_listing = pd.DataFrame(df_, columns= df_.columns[sale_feat_index])

In [103]:
df_listing

,listPrice,latitude,longitude,DOM,avg_sqft,ppsqft,bathtobed_ratio,listPrice_by_area,listPrice_by_city,originalPrice_99900000,class_CondoProperty,city_Kitchener,city_Markham,city_RichmondHill,numBathrooms_1,numBathrooms_2,numBathrooms_3,numBathrooms_4,numBathrooms_5,numBedrooms_2,numBedrooms_3,numBedrooms_4,numBedrooms_5,style_2Storey,style_Bungalow
0,1999000.0,43.729052,-79.426974,NaN,NaN,NaN,1.0,1424204.466,1424204.466,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,1


In [130]:
x = pd.DataFrame(columns = ['a', 'a', 'b'])

In [131]:
# asdf = asdf.drop(['a'], axis = 1)
print(set(x.columns))
x.columns

list(x.columns).count('a')

list(x.columns) != set(x.columns)

t = set([y for y in x if list(x.columns).count(y) > 1])

asdf = x.drop(t, axis = 1)

asdf

{'a', 'b'}


,b


In [137]:
import sklearn
import flask 
import flask_cors
import lightgbm
print(pd.__version__)
print(np.__version__)
print(sklearn.__version__)
print(flask.__version__)
print(flask_cors.__version__)
print(lightgbm.__version__)

1.4.4
1.21.5
1.0.2
2.3.2
3.0.10
3.3.5
